<div class='alert alert-warning'>

# JupyterLite warning

- Running the **scikit-plots 0.5.dev0** interactive examples in JupyterLite is experimental and may not always work as expected.
- With high load times especially on low-resource platforms, and the version of scikit-plots might not be in sync with the one you are browsing the documentation for.
- If you encounter any issues, please report them on the [scikit-plots issue tracker](https://github.com/scikit-plots/scikit-plots/issues/new/choose).
- `micropip/piplite/pip` use `%pip` in JupyterLite instead of `pip` or `!pip`

```python
## Installing the dependencies first, and then scikit-plots from Anaconda.org.
import piplite; await piplite.install(  # or micropip
   'scikit-plots==0.5.dev0',            # Download scikit-plots *pyodide_20XX_0_wasm32.whl
   index_urls='https://pypi.anaconda.org/scikit-plots-wheels-staging-nightly/simple',
); import sklearn; import scikitplot as sp; sp.show_versions();
 ```

</div>

Plug in ``pdfplumber`` as a custom PDF backend:


In [ ]:
import pdfplumber
from pathlib import Path
from scikitplot.corpus._readers._custom import CustomReader

def pdfplumber_extract(path, **kw):
    with pdfplumber.open(path) as pdf:
        return [
            {"text": page.extract_text() or "", "page_number": i}
            for i, page in enumerate(pdf.pages)
        ]

reader = CustomReader(
    input_path=Path("report.pdf"),
    extractor=pdfplumber_extract,
)
docs = list(reader.get_documents())

Register globally and use via factory:


In [ ]:
CustomReader.register(
    name="PdfPlumberReader",
    extensions=[".pdf"],
    extractor=pdfplumber_extract,
    default_source_type=SourceType.RESEARCH,
)
reader = DocumentReader.create(Path("report.pdf"))
docs = list(reader.get_documents())

Custom audio transcription (e.g. a proprietary ASR API):


In [ ]:
def my_asr(path, language="en", **kw):
    result = my_asr_client.transcribe(path, lang=language)
    return [
        {"text": seg.text, "timecode_start": seg.start, "timecode_end": seg.end}
        for seg in result.segments
    ]

CustomReader.register(
    name="MyASRReader",
    extensions=[".mp3", ".wav", ".flac"],
    extractor=my_asr,
    reader_kwargs={"language": "de"},
    default_source_type=SourceType.PODCAST,
)

Non-filesystem source (validate_file=False):


In [ ]:
def stream_extractor(path, **kw):
    # path is a synthetic Path wrapping a stream identifier
    data = fetch_from_stream(str(path))
    return data.decode("utf-8")

reader = CustomReader(
    input_path=Path("stream://channel/42"),
    extractor=stream_extractor,
    validate_file=False,
)